# 03 | สำรวจคะแนนและประเด็นจากรีวิว Amari Phuket

รายงานนี้ตอบคำถามห้าข้อด้วยกราฟเจ็ดรูป หน่วยวิเคราะห์คือรีวิวหนึ่งรายการ แกนเวลาใช้ **วันที่เขียนรีวิว** ดูการทำความสะอาดใน notebook 02 และข้อเสนอแนะใน [Insights](../docs/insights.md)

In [ ]:
from pathlib import Path
import hashlib

import pandas as pd

# รองรับการเปิดจากโฟลเดอร์โปรเจกต์หรือจาก notebooks/
working_dir = Path.cwd()
root = working_dir.parent if working_dir.name == "notebooks" else working_dir
if not (root / "README.md").is_file() or not (root / "notebooks").is_dir():
    raise FileNotFoundError("Open the notebook from the portfolio folder or its notebooks/ folder.")

raw_name = "samplehotel - Sheet1.csv"
raw_path = root / "data" / "raw" / raw_name
if not raw_path.is_file():
    raise FileNotFoundError(f"Missing data/raw/{raw_name}. See README.md for setup.")
source_hash = hashlib.sha256(raw_path.read_bytes()).hexdigest()
print(f"Source: {raw_path.relative_to(root)}")
print(f"SHA-256: {source_hash}")

In [ ]:
processed_dir = root / "data" / "processed"
cleaned_path = processed_dir / "cleaned_hotel_reviews.csv"
manifest_path = processed_dir / "source.json"

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

if not cleaned_path.is_file() or not manifest_path.is_file():
    raise FileNotFoundError("Run notebook 02 before EDA.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest["source_file"] != raw_name or manifest["source_sha256"] != source_hash:
    raise ValueError("Raw data changed. Run notebook 02 again before EDA.")

reviews = pd.read_csv(cleaned_path, keep_default_na=False,
                      parse_dates=["clean_written_date", "review_month_start"])
assert len(reviews) == manifest["rows"]
assert reviews["rating"].isin(range(1, 6)).all()
low_group = reviews["rating"].le(3)
total = len(reviews)
figure_dir = root / "outputs" / "eda"
figure_dir.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.spines.top": False,
    "axes.spines.right": False, "axes.titleweight": "bold",
    "axes.labelcolor": "#253548", "text.color": "#253548",
})
teal, navy, muted, coral = "#167C80", "#274566", "#7C8899", "#B34C36"

def publish_figure(fig, filename):
    """บันทึกกราฟและแสดงใน Notebook"""
    fig.savefig(figure_dir / filename, dpi=160, facecolor="white")
    plt.show()
    plt.close(fig)

## 1. ภาพรวมความพึงพอใจ

กราฟแท่งแสดงจำนวนและสัดส่วนในแต่ละคะแนน ดูทั้งค่าเฉลี่ยและสัดส่วน 1–3 คะแนนเพื่อเห็นด้านที่ค่าเฉลี่ยซ่อนไว้ ผู้ที่เขียนรีวิวอาจมีประสบการณ์ต่างจากแขกที่ไม่ได้เขียน

In [ ]:
rating_counts = reviews["rating"].value_counts().reindex(range(1, 6), fill_value=0)
display(pd.Series({"reviews": total, "mean": reviews["rating"].mean(),
                   "median": reviews["rating"].median(), "low_reviews": low_group.sum()}))
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
bars = ax.bar(rating_counts.index, rating_counts, color=[coral] * 3 + [muted, teal], width=.65)
ax.bar_label(bars, labels=[f"{n:,} ({n/total:.1%})" for n in rating_counts], padding=4)
ax.set(title=f"Guest rating distribution | n = {total:,}", xlabel="Guest rating",
       ylabel="Reviews", xticks=range(1, 6), ylim=(0, rating_counts.max() * 1.16))
publish_figure(fig, "01_rating_distribution.png")

รีวิวส่วนใหญ่ให้คะแนน 5 จึงใช้รีวิว 1–3 อ่านหาปัญหาควบคู่กับการรายงานค่าเฉลี่ย

## 2. คะแนนตามประเภทนักเดินทาง

จุดแสดงค่าเฉลี่ยบนแกน 1–5 เท่ากันทุกกลุ่ม ป้ายบอกจำนวนรีวิว โปรดดูสัดส่วนคะแนนต่ำประกอบ ตารางเก็บ `Unknown` และกลุ่มเล็กไว้เพื่อให้เห็นความครอบคลุม

In [ ]:
segment = (
    reviews.assign(low=low_group)
    .groupby("travel_type")
    .agg(n=("rating", "size"), mean=("rating", "mean"), low=("low", "sum"))
    .sort_values("n", ascending=False)
)
segment["low_pct"] = 100 * segment["low"] / segment["n"]
display(segment.round(3))
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
labels = [f"{label}  (n={n:,})" for label, n in segment["n"].items()]
ax.scatter(segment["mean"], labels,
           c=[coral if x == "Family" else muted if x == "Unknown" else teal
              for x in segment.index], s=54)
for i, mean in enumerate(segment["mean"]):
    ax.annotate(f"{mean:.2f}", (mean, i), xytext=(-8, 0), textcoords="offset points",
                ha="right", va="center")
ax.invert_yaxis()
ax.grid(axis="x", alpha=.2)
ax.set(title="Average rating by traveler type", xlabel="Mean guest rating (1–5)",
       xlim=(1, 5.2), xticks=range(1, 6))
publish_figure(fig, "02_traveler_ratings.png")

Family มีรีวิวมากพอให้เริ่มอ่านเชิงคุณภาพ และมีสัดส่วนคะแนนต่ำสูงกว่า Couples การจัดอันดับจากกลุ่มเล็กอย่าง Business มีความไม่แน่นอนสูง ส่วน `Unknown` ไม่สามารถระบุประเภทนักเดินทางได้

## 3. คะแนนและจำนวนรีวิวตามเวลา

กราฟรายปีแยกจำนวนกับคะแนนไว้คนละแผง เพื่อให้แต่ละแกนอ่านง่าย ตรวจจำนวนเดือนที่พบและวันที่แรก/สุดท้ายของแต่ละปี เปรียบเทียบช่วง ม.ค.–ก.ค. เฉพาะปีที่มีรีวิวครบทั้งเจ็ดเดือนเป็นการตรวจความไวต่อขอบเขตข้อมูล

In [ ]:
annual = reviews.groupby("review_year").agg(
    n=("rating", "size"), mean=("rating", "mean"),
    months_seen=("review_month", "nunique"),
    first_written=("clean_written_date", "min"), last_written=("clean_written_date", "max"),
)
display(annual.round({"mean": 3}))
jan_jul = reviews.loc[reviews["review_month"].between(1, 7)]
comparable = jan_jul.groupby("review_year").agg(
    n=("rating", "size"), mean=("rating", "mean"), months_seen=("review_month", "nunique")
)
display(comparable.loc[comparable["months_seen"].eq(7)].round(3))
fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 5.6), sharex=True, layout="constrained")
bars = top.bar(annual.index, annual["n"], width=.55, color=teal)
top.bar_label(bars, padding=3)
top.set(title=f"Written review volume and rating by year | n = {total:,}",
        ylabel="Reviews", ylim=(0, annual["n"].max() * 1.15))
bottom.plot(annual.index, annual["mean"], marker="o", color=navy)
for year, mean in annual["mean"].items():
    bottom.annotate(f"{mean:.2f}", (year, mean), xytext=(0, 7),
                    textcoords="offset points", ha="center")
bottom.set(xlabel="Year written", ylabel="Mean rating (1–5)", xticks=annual.index,
           ylim=(1, 5.3), yticks=range(1, 6))
publish_figure(fig, "03_reviews_by_year.png")

In [ ]:
monthly = reviews.groupby("review_month_start")["rating"].agg(n="size", mean="mean")
months = pd.date_range(monthly.index.min(), monthly.index.max(), freq="MS")
monthly = monthly.reindex(months).rename_axis("month_written")
monthly["n"] = monthly["n"].fillna(0).astype(int)
display(monthly.tail(12).round(3))
fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 5.6), sharex=True, layout="constrained")
top.bar(monthly.index, monthly["n"], color=teal, width=20)
top.set(title=f"Written review volume and rating by month | n = {total:,}", ylabel="Reviews")
bottom.plot(monthly.index, monthly["mean"], marker="o", markersize=4, color=navy)
small = monthly.loc[monthly["n"].between(1, 9)]
if not small.empty:
    bottom.scatter(small.index, small["mean"], facecolor="white", edgecolor=coral,
                   zorder=3, label="1–9 reviews")
    bottom.legend(loc="lower left", frameon=False)
bottom.set(xlabel="Month written", ylabel="Mean rating (1–5)", ylim=(1, 5.3), yticks=range(1, 6))
bottom.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
bottom.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
publish_figure(fig, "04_reviews_by_month.png")

ค่าเฉลี่ยปี 2022 ต่ำกว่าเดิมแม้จำกัดช่วงเดือนที่เปรียบเทียบได้ กราฟนับรีวิวที่เขียน เดือนที่ไม่มีรีวิวเว้นจุดค่าเฉลี่ยไว้ และวงกลมโปร่งแสดงเดือนที่มีข้อมูลต่ำกว่า 10 รีวิว

## 4. ช่องทางเผยแพร่และประเทศผู้รีวิว

กราฟช่องทางเน้นปริมาณรีวิวพร้อมแสดงค่าเฉลี่ยในตาราง `Other` เป็นค่าจากต้นทางและไม่ทราบช่องทางย่อย กราฟประเทศแสดง `Unknown` แยกจาก 5 ประเทศที่มีรีวิวมากที่สุด และรวมประเทศที่ระบุชื่ออื่นไว้เพื่อให้ยอดรวมเท่ากับทั้งไฟล์

In [ ]:
platform = (reviews.groupby("publish_platform")["rating"]
            .agg(n="size", mean="mean").sort_values("n", ascending=False))
display(platform.round(3))
fig, ax = plt.subplots(figsize=(8, 3.6), layout="constrained")
bars = ax.barh(platform.index, platform["n"], color=teal)
ax.bar_label(bars, labels=[f"{n:,} | mean {mean:.2f}" for n, mean in platform.itertuples(index=False)], padding=5)
ax.invert_yaxis()
ax.set(title=f"Review publishing channels | n = {total:,}", xlabel="Reviews",
       xlim=(0, platform["n"].max() * 1.30))
publish_figure(fig, "05_publishing_channels.png")

In [ ]:
country = (reviews.groupby("clean_country")["rating"]
           .agg(n="size", mean="mean").sort_values("n", ascending=False))
known = country.drop(index="Unknown", errors="ignore")
country_counts = known["n"].head(5).copy()
if len(known) > 5:
    country_counts.loc["All other known countries"] = known["n"].iloc[5:].sum()
if "Unknown" in country.index:
    country_counts = pd.concat([country.loc[["Unknown"], "n"], country_counts])
display(country.head(10).round(3))
fig, ax = plt.subplots(figsize=(8, 4.6), layout="constrained")
bars = ax.barh(country_counts.index, country_counts,
               color=[muted if x == "Unknown" else teal for x in country_counts.index])
ax.bar_label(bars, labels=[f"{n:,} ({n/total:.1%})" for n in country_counts], padding=4)
ax.invert_yaxis()
ax.set(title=f"Reviewer country coverage | n = {total:,}", xlabel="Reviews",
       xlim=(0, country_counts.max() * 1.23))
publish_figure(fig, "06_reviewer_countries.png")

ช่องทางสองกลุ่มมีคะแนนเฉลี่ยใกล้กัน ข้อมูลประเทศขาดหายเกือบครึ่ง จึงควรตรวจความครบถ้วนก่อนใช้คะแนนตามประเทศจัดอันดับตลาด

## 5. ประเด็นปัญหาที่พบในรีวิวคะแนน 1–3

ใช้กฎค้น **วลีที่บ่งชี้ปัญหา** ในหัวข้อและเนื้อหา ฉบับนี้ใช้วลีไทย/อังกฤษเพื่อหาเบาะแสเบื้องต้น เช่น “พนักงานไม่แนะนำ” หรือ “dirty linen” การนับใช้หนึ่งครั้งต่อรีวิวต่อธีม และหนึ่งรีวิวเข้าได้หลายธีม ตารางเทียบจำนวนการพบในข้อความที่มี/ไม่มีธงตัด เพื่อเห็นว่าการครอบคลุมต่างกันหรือไม่

กฎจับเพียงสำนวนที่กำหนด อาจพลาดการบ่นภาษาที่สาม ถ้อยคำอ้อม ๆ หรือข้อความที่ถูกตัด และอาจจับข้อความที่ยกตัวอย่างหรือกล่าวถึงปัญหาที่ได้รับการแก้แล้ว การตรวจซ้ำทำได้โดยใช้ตารางตำแหน่งแถวเพื่อตามอ่านข้อความใน CSV

In [ ]:
low_reviews = reviews.loc[low_group].copy()
screen_text = (low_reviews["title"].str.cat(low_reviews["text"], sep=" | ")
               .str.casefold().str.replace(r"\s+", " ", regex=True))
complaint_patterns = {
    "Service and staff": (
        r"\b(?:rude|arrogant|dishonest|unhelpful|obnoxious) staff\b|\bstaff (?:were|was) (?:rude|unhelpful)\b"
        r"|\b(?:poor|bad|slow|horrible|terrible) (?:customer |room |quality )?service\b"
        r"|\b(?:understaffed|staff training|lack of staff|room service is very slow)\b"
        r"|พนักงานไม่แนะนำ|บริการไม่ทั่วถึง|บริการอาจจะไม่ค่อยทั่วถึง|ไม่ค่อยประทับใจ.{0,40}บริการ"
        r"|บริการของพนักงาน.{0,30}ไม่ค่อยประทับใจ|พนักงานบางท่านควรเป็นมิตร"
    ),
    "Room condition and maintenance": (
        r"\b(?:dated|tired|outdated|run down|run-down) (?:hotel|room|property)\b"
        r"|\b(?:hotel|property|room|room decor) (?:is |was )?(?:old|dated|tired|run down|run-down)\b"
        r"|\b(?:old hotel|old property|wear and tear|lacks upkeep|not well maintained|requires? maintenance|required maintenance)\b"
        r"|ท่อตัน|อุปกรณ์ชำรุด|ประตูห้องน้ำ.{0,12}มีปัญหา|แอร์ตั้งไว้ให้ตัด|ห้องพัก.{0,20}เก่า"
    ),
    "Access and internal transport": (
        r"\b(?:wait|waiting|waited)\b[^.!?]{0,65}\b(?:buggy|shuttle|cart)\b"
        r"|\b(?:lots of steps|not very accessible|had to climb|no lift|no elevator)\b"
        r"|\b(?:far from the room|difficult to travel with kids|don't provide shuttle full day)\b"
        r"|ไม่มีลิฟ|ทางเดินชัน|เดินขึ้นลงสามชั้น"
    ),
    "Cleanliness": (
        r"\b(?:dirty|stained) (?:room|rooms|linen|sheets|bed sheets|bathroom)\b"
        r"|\b(?:room|rooms|linen|sheets|bathroom) (?:is|are|was|were) (?:dirty|stained)\b"
        r"|\b(?:unhygienic|mou?ld in (?:our|the) bathroom|not so clean in the bathroom)\b"
        r"|ความสะอาดไม่ค่อยดี|ห้อง.{0,10}ไม่สะอาด"
    ),
    "Food and breakfast": (
        r"\b(?:breakfast (?:was|is) (?:chaotic|cold|poor|bad)|food quality (?:was|is) (?:bad|poor)"
        r"|food quantity should be more|(?:poor|bad|cold) breakfast|worse experience in food)\b"
        r"|อาหารเช้าไม่อร่อย|อาหารไม่อร่อย"
    ),
    "Price and value": (
        r"\b(?:overpriced|inflated pricing|way too expensive|not worth (?:the money|of money)|poor value)\b"
        r"|ไม่คุ้มค่า|ราคาแพงเกิน"
    ),
}
flags = pd.DataFrame({theme: screen_text.str.contains(pattern, regex=True)
                      for theme, pattern in complaint_patterns.items()})
clipped = low_reviews["possible_truncation"]
has_thai = screen_text.str.contains(r"[\u0E00-\u0E7F]")
theme_counts = pd.DataFrame({
    "all_matches": flags.sum(),
    "with_truncation_flag": flags.loc[clipped].sum(),
    "without_truncation_flag": flags.loc[~clipped].sum(),
    "has_thai_script": flags.loc[has_thai].sum(),
}).sort_values("all_matches", ascending=False, kind="stable")

display(theme_counts)
print(f"Low reviews: {len(low_reviews)} | Any theme: {flags.any(axis=1).sum()} | "
      f"Truncation flag: {clipped.sum()} | Contains Thai script: {has_thai.sum()}")
# แถวข้อมูลนับจาก 1 โดยไม่นับหัวตาราง ใช้ iloc[position - 1] ดูฉบับเต็มที่มีอยู่
match_audit = flags.rename_axis("row_index").reset_index()
match_audit["data_row"] = match_audit.pop("row_index") + 1
display(pd.Series({theme: match_audit.loc[match_audit[theme], "data_row"].tolist()
                   for theme in theme_counts.index}, name="data_rows"))

fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
bars = ax.barh(theme_counts.index, theme_counts["all_matches"], color=coral)
ax.bar_label(bars, padding=5)
ax.invert_yaxis()
ax.set(title=f"Complaint phrases in low ratings | n = {len(low_reviews):,} reviews",
       xlabel="Reviews matching the theme", xlim=(0, max(1, theme_counts["all_matches"].max()) * 1.16))
publish_figure(fig, "07_low_rating_complaints.png")

ใช้ลำดับจากกราฟเพื่อเลือกตัวอย่างอ่านก่อน พร้อมสุ่มอ่านข้อความที่กฎจับไม่ได้ และตรวจข้ามกับบันทึกข้อร้องเรียนของโรงแรมก่อนจัดลำดับงานบริการ สัดส่วนการพบวลีขึ้นอยู่กับชุดคำที่เลือกและคุณภาพข้อความ